# H/C peak-to-atom assignment

Propose atom assignments for measured peaks and a known structure.

This notebook displays recorded real evaluation results. Model inference is opt-in and never trains parameters.

In [1]:
import json, os, sys, subprocess, tempfile
from pathlib import Path
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'openmolspec_bench').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from the cloned OpenMolSpec frontend repository')
task_id = 'nmr.assignment'
example_input = {'smiles': 'CCO', 'nucleus': '13C', 'peaks_ppm': [18.2, 58.3], 'tolerance_ppm': 5.0}
print(json.dumps(example_input, indent=2, ensure_ascii=False))

{
  "smiles": "CCO",
  "nucleus": "13C",
  "peaks_ppm": [
    18.2,
    58.3
  ],
  "tolerance_ppm": 5.0
}


## Recorded evaluation

Results retain their sample size, conditions and source file; an execution pass does not imply accuracy.

In [2]:
index = json.loads((root / 'documentation/assets/evaluation_index.json').read_text())
recorded = index.get(task_id, [])
print(json.dumps(recorded or {'status': 'not_evaluated', 'reason': 'Required compatible assets or adapter unavailable'}, indent=2, ensure_ascii=False))

[
  {
    "task": "nmr.assignment",
    "status": "passed",
    "metrics": {
      "molecules": 32,
      "matched_peaks": 488,
      "matched_peak_fraction_macro": 0.9392376720501721,
      "matched_residual_mae_ppm": 0.8991307922920895,
      "assignment_accuracy": null
    },
    "selection": {
      "read": 32,
      "missing_modality": 0,
      "duplicate_structure": 0,
      "exceeds_model_capacity": 0
    },
    "purpose": "real_peak_assignment_execution_not_atom_accuracy_benchmark",
    "limitations": [
      "No experimental atom assignment ground truth supplied.",
      "Symmetry is topological; conformer/solvent effects can change observed equivalence.",
      "Residual and coverage are diagnostics, not assignment accuracy.",
      "Training overlap unverified; frozen model is not solvent-conditioned."
    ],
    "source_file": "assignment_carbon32_20261116.json",
    "sample_limit": 32
  }
]


## Optional frozen inference

Set `OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE=1` and configure the workspace and model runtime. Missing weights cause a skip/error, never training.

In [3]:
if os.environ.get('OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE') == '1':
    runtime = os.environ.get('OPENMOLSPEC_PYTHON', sys.executable)
    workspace = os.environ.get('OPENMOLSPEC_BENCH_ROOT', str(root.parent))
    with tempfile.TemporaryDirectory(prefix='openmolspec-notebook-') as directory:
        command = [runtime, '-m', 'openmolspec_bench', 'run', '--workspace', workspace, '--tasks', task_id, '--limit', '2', '--output', str(Path(directory) / 'result.json')]
        subprocess.run(command, cwd=root, check=True)
        print((Path(directory) / 'result.json').read_text())
else:
    print('Model execution disabled; recorded evaluation displayed above. No training performed.')

Model execution disabled; recorded evaluation displayed above. No training performed.
